# Load BTC data into Supabase
Run the cells **top to bottom** (Runtime → Run all works too). It makes the tables, then loads your Colab download.
Safe to run again: existing rows are skipped, only new rows are added.

## 1. Install the database tool

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y postgresql-client > /dev/null && psql --version

## 2. Upload `delta_btc_data.zip`
Click **Choose files** and pick the zip you downloaded earlier.

In [ ]:
import os, zipfile
from google.colab import files
os.makedirs('/content/data', exist_ok=True)
up = files.upload()
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('/content/data')
    else:
        os.replace(name, '/content/data/' + name)
need = ['btc_spot_1h.csv', 'btc_options_products.csv', 'btc_options_1h.csv']
missing = [f for f in need if not os.path.exists('/content/data/' + f)]
print('Missing files:', missing) if missing else print('All 3 files found:', need)

## 3. Connection details
In Supabase: open your project → **Connect** (top bar) → **Session pooler**. Copy the **Host** and **User** (looks like `postgres.abcdefgh`). 
The password is the database password you set when you created the project. It is typed into a hidden box and is not saved in this notebook.

In [ ]:
import getpass, os
os.environ['PGHOST'] = input('Host (aws-0-...pooler.supabase.com): ').strip()
os.environ['PGUSER'] = input('User (postgres.xxxxxxxx): ').strip()
os.environ['PGPASSWORD'] = getpass.getpass('Database password: ')
os.environ['PGPORT'] = '5432'
os.environ['PGDATABASE'] = 'postgres'
os.environ['PGSSLMODE'] = 'require'
!psql -c "select 'connected' as status"

## 4. Create the tables and views

In [ ]:
%%writefile /content/20261018000000_btc_market_data.sql
-- BTC market data: hourly spot, Delta option contracts, Delta option hourly candles.
-- All timestamps are stored as timestamptz (UTC). IST is derived in views.
-- Public market data only: anyone with the anon/publishable key can read, nobody but the
-- service role / postgres user can write.

create table if not exists public.btc_spot_1h (
  ts      timestamptz primary key,          -- candle open time (UTC)
  open    numeric(14,2) not null,
  high    numeric(14,2) not null,
  low     numeric(14,2) not null,
  close   numeric(14,2) not null,
  volume  numeric(24,4),
  check (high >= low)
);
comment on table public.btc_spot_1h is 'BTC hourly spot candles (Delta BTCUSD). ts is the candle open in UTC.';

create table if not exists public.option_contracts (
  symbol           text primary key,        -- e.g. C-BTC-83200-181026
  option_type      text not null check (option_type in ('call','put')),
  strike           numeric(12,2) not null,
  expiry_ts        timestamptz not null,    -- Delta daily options expire 12:00 UTC (5:30 PM IST)
  spot_day_before  numeric(14,2)            -- BTC spot noted by the downloader the day before expiry
);
comment on table public.option_contracts is 'Delta Exchange India BTC option contracts, one row per symbol.';
create index if not exists option_contracts_expiry_idx on public.option_contracts (expiry_ts, option_type, strike);

create table if not exists public.option_candles_1h (
  symbol  text not null references public.option_contracts (symbol) on delete cascade,
  ts      timestamptz not null,             -- candle open time (UTC)
  open    numeric(16,6) not null,
  high    numeric(16,6) not null,
  low     numeric(16,6) not null,
  close   numeric(16,6) not null,
  volume  numeric(24,4),
  kind    text not null default 'mark' check (kind in ('mark','trade')),  -- mark = mark-price candles, not executable prices
  primary key (symbol, ts),
  check (high >= low)
);
comment on table public.option_candles_1h is 'Hourly candles per option contract. kind=mark means mark price (not a traded price).';
create index if not exists option_candles_ts_idx on public.option_candles_1h (ts);

-- Read-only public access through row level security.
alter table public.btc_spot_1h       enable row level security;
alter table public.option_contracts  enable row level security;
alter table public.option_candles_1h enable row level security;

drop policy if exists "public read" on public.btc_spot_1h;
drop policy if exists "public read" on public.option_contracts;
drop policy if exists "public read" on public.option_candles_1h;
create policy "public read" on public.btc_spot_1h       for select to anon, authenticated using (true);
create policy "public read" on public.option_contracts  for select to anon, authenticated using (true);
create policy "public read" on public.option_candles_1h for select to anon, authenticated using (true);

revoke all on public.btc_spot_1h, public.option_contracts, public.option_candles_1h from anon, authenticated;
grant select on public.btc_spot_1h, public.option_contracts, public.option_candles_1h to anon, authenticated;

In [ ]:
%%writefile /content/20261018000100_btc_views.sql
-- Analysis views. security_invoker makes them obey the caller's row level security.

-- Every option candle with its contract details, IST time and hours to expiry.
create or replace view public.v_option_candles with (security_invoker = on) as
select
  c.symbol,
  k.option_type,
  k.strike,
  k.expiry_ts,
  c.ts,
  (c.ts at time zone 'Asia/Kolkata')                           as ts_ist,
  extract(epoch from (k.expiry_ts - c.ts)) / 3600.0            as hours_to_expiry,
  c.open, c.high, c.low, c.close, c.volume, c.kind
from public.option_candles_1h c
join public.option_contracts k using (symbol);

-- The trading day: 5:30 PM IST to 5:30 PM IST (12:00 UTC to 12:00 UTC). Complete 24-candle windows only.
-- window_end is the expiry that closes the window.
create or replace view public.btc_daily_530 with (security_invoker = on) as
with w as (
  select
    -- computed in UTC explicitly so the result does not depend on the session time zone
    (date_trunc('day', (ts at time zone 'UTC') - interval '12 hours') + interval '12 hours') at time zone 'UTC' as window_start,
    ts, open, high, low, close
  from public.btc_spot_1h
)
select
  window_start,
  window_start + interval '1 day'                                   as window_end,
  extract(isodow from (window_start + interval '1 day') at time zone 'UTC')::int as end_isodow,  -- 1=Mon .. 7=Sun
  (array_agg(open  order by ts asc))[1]                             as open,
  max(high)                                                         as high,
  min(low)                                                          as low,
  (array_agg(close order by ts desc))[1]                            as close,
  max(high) - min(low)                                              as range_pts,
  (max(high) - min(low)) / nullif((array_agg(open order by ts asc))[1], 0) * 100 as range_pct,
  abs((array_agg(close order by ts desc))[1] - (array_agg(open order by ts asc))[1]) as abs_move_pts,
  count(*)                                                          as candles
from w
group by window_start
having count(*) = 24;

-- Average hourly range by IST hour, split weekday / weekend (by IST date).
create or replace view public.btc_hourly_activity with (security_invoker = on) as
select
  extract(hour from (ts at time zone 'Asia/Kolkata'))::int          as ist_hour,
  (extract(isodow from (ts at time zone 'Asia/Kolkata')) >= 6)      as is_weekend,
  count(*)                                                          as candles,
  avg((high - low) / nullif(open, 0) * 100)                         as avg_range_pct,
  avg(high - low)                                                   as avg_range_pts,
  avg(volume)                                                       as avg_volume
from public.btc_spot_1h
group by 1, 2;

-- One row per expiry: what the data covers.
create or replace view public.v_data_coverage with (security_invoker = on) as
select
  (select min(ts) from public.btc_spot_1h)            as spot_from,
  (select max(ts) from public.btc_spot_1h)            as spot_to,
  (select count(*) from public.btc_spot_1h)           as spot_rows,
  (select min(expiry_ts) from public.option_contracts) as first_expiry,
  (select max(expiry_ts) from public.option_contracts) as last_expiry,
  (select count(*) from public.option_contracts)      as contracts,
  (select count(*) from public.option_candles_1h)     as option_candles;

revoke all on public.v_option_candles, public.btc_daily_530, public.btc_hourly_activity, public.v_data_coverage from anon, authenticated;
grant select on public.v_option_candles, public.btc_daily_530, public.btc_hourly_activity, public.v_data_coverage to anon, authenticated;

In [ ]:
!for f in /content/2*.sql; do echo "== $f"; psql -v ON_ERROR_STOP=1 -q -f $f || exit 1; done; echo done

## 5. Load the data (takes about 1 to 3 minutes)

In [ ]:
%%writefile /content/data/load_csv.sql
-- Loads the three CSV files from the Colab download into the tables. Safe to re-run:
-- existing rows are left alone and only new rows are added.
-- Run with psql from the folder that holds the CSV files (see load/README.md).
\set ON_ERROR_STOP on
set statement_timeout = 0;
set timezone = 'UTC';
\timing off
begin;

create temp table s_spot (time_utc text, time_ist text, open numeric, high numeric, low numeric, close numeric, volume numeric);
\copy s_spot from 'btc_spot_1h.csv' with (format csv, header true)
insert into public.btc_spot_1h (ts, open, high, low, close, volume)
select (time_utc || ' UTC')::timestamptz, open, high, low, close, volume from s_spot
on conflict (ts) do update set open = excluded.open, high = excluded.high, low = excluded.low,
  close = excluded.close, volume = excluded.volume
  where (public.btc_spot_1h.open, public.btc_spot_1h.high, public.btc_spot_1h.low, public.btc_spot_1h.close)
        is distinct from (excluded.open, excluded.high, excluded.low, excluded.close);  -- fixes a candle that was unfinished when first saved

create temp table s_prod (symbol text, type text, strike numeric, expiry_utc text, expiry_ts bigint, spot_day_before numeric);
\copy s_prod from 'btc_options_products.csv' with (format csv, header true)
insert into public.option_contracts (symbol, option_type, strike, expiry_ts, spot_day_before)
select symbol, type, strike, to_timestamp(expiry_ts), spot_day_before from s_prod
on conflict (symbol) do nothing;

create temp table s_opt (symbol text, type text, strike numeric, expiry_utc text, time_utc text, time_ist text,
  hours_to_expiry numeric, open numeric, high numeric, low numeric, close numeric, volume numeric, source text);
\copy s_opt from 'btc_options_1h.csv' with (format csv, header true)
insert into public.option_candles_1h (symbol, ts, open, high, low, close, volume, kind)
select symbol, (time_utc || ' UTC')::timestamptz, open, high, low, close, volume,
       case when source = 'trade' then 'trade' else 'mark' end
from s_opt
on conflict (symbol, ts) do nothing;

commit;
analyze public.btc_spot_1h; analyze public.option_contracts; analyze public.option_candles_1h;
select * from public.v_data_coverage;

In [ ]:
!cd /content/data && psql -v ON_ERROR_STOP=1 -q -f load_csv.sql

## 6. Check
The last table should show about **8,800 spot rows, 20,440 contracts and 520,441 option candles**.

In [ ]:
!psql -c "select * from public.v_data_coverage"